# ML Challenge - Fault Detection
binary classification on sensor data (47 features) to detect faulty vs normal devices

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## load data

In [2]:
train = pd.read_csv('TRAIN.csv')
test = pd.read_csv('TEST.csv')

print(train.shape, test.shape)
print('missing train:', train.isnull().sum().sum())
print('missing test:', test.isnull().sum().sum())
train['Class'].value_counts()

(43776, 48) (10944, 48)
missing train: 0
missing test: 0


Class
0    26465
1    17311
Name: count, dtype: int64

## preprocessing

In [3]:
# drop dupes
before = len(train)
train = train.drop_duplicates()
print(f'dropped {before - len(train)} duplicates, now {len(train)}')

test_ids = test['ID']
feature_cols = [c for c in train.columns if c != 'Class']
test_feature_cols = [c for c in test.columns if c not in ('ID', 'Class')]

dropped 738 duplicates, now 43038


## feature engineering

In [4]:
def make_features(df, feat_cols):
    data = df[feat_cols].copy()
    
    # row-wise stats (using numpy for speed)
    vals = data[feat_cols].values
    data['feat_mean'] = vals.mean(axis=1)
    data['feat_std'] = vals.std(axis=1)
    data['feat_min'] = vals.min(axis=1)
    data['feat_max'] = vals.max(axis=1)
    data['feat_range'] = data['feat_max'] - data['feat_min']
    
    # log transform heavy-tailed features
    for col in ['F30','F31','F32','F33','F34','F35','F36','F37','F38']:
        data[f'{col}_log'] = np.log1p(data[col])
    
    # interaction terms between top correlated features
    data['F01_x_F09'] = data['F01'] * data['F09']
    data['F01_x_F29'] = data['F01'] * data['F29']
    data['F19_div_F21'] = data['F19'] / (data['F21'] + 1e-8)
    data['F05_div_F06'] = data['F05'] / (data['F06'] + 1e-8)
    data['F25_x_F27'] = data['F25'] * data['F27']
    data['F07_x_F08'] = data['F07'] * data['F08']
    
    # group sums
    data['sum_F01_F09'] = data[['F01','F02','F03','F04','F05','F06','F07','F08','F09']].sum(axis=1)
    data['sum_F10_F18'] = data[['F10','F11','F12','F13','F14','F15','F16','F17','F18']].sum(axis=1)
    data['sum_F19_F29'] = data[['F19','F20','F21','F22','F23','F24','F25','F26','F27','F28','F29']].sum(axis=1)
    data['sum_F30_F38'] = data[['F30','F31','F32','F33','F34','F35','F36','F37','F38']].sum(axis=1)
    data['sum_F39_F47'] = data[['F39','F40','F41','F42','F43','F44','F45','F46','F47']].sum(axis=1)
    
    return data

In [5]:
X_eng = make_features(train, feature_cols)
X_test_eng = make_features(test, test_feature_cols)

X = X_eng.values
y = train['Class'].values
X_test = X_test_eng.values

print(f'features: {X.shape[1]}, train: {X.shape[0]}, test: {X_test.shape[0]}')

features: 72, train: 43038, test: 10944


## models
reduced n_estimators + higher learning rate to keep things fast on colab. lgbm is super fast anyway, xgb hist mode also quick enough

In [6]:
scale_pos = y[y==0].shape[0] / y[y==1].shape[0]

xgb1 = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.6,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos,
    min_child_weight=5,
    gamma=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    tree_method='hist'
)

lgbm1 = LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.6,
    reg_alpha=0.1,
    reg_lambda=1.0,
    is_unbalance=True,
    min_child_samples=20,
    num_leaves=63,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm2 = LGBMClassifier(
    n_estimators=400,
    max_depth=10,
    learning_rate=0.08,
    subsample=0.75,
    colsample_bytree=0.7,
    reg_alpha=0.05,
    reg_lambda=0.5,
    is_unbalance=True,
    min_child_samples=30,
    num_leaves=127,
    random_state=123,
    n_jobs=-1,
    verbose=-1
)

print('models ready')

models ready


## quick CV check
3-fold instead of 5, just to get a rough idea. no need to waste 40 min on CV

In [7]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, mdl in [('XGBoost', xgb1), ('LightGBM-1', lgbm1), ('LightGBM-2', lgbm2)]:
    scores = cross_val_score(mdl, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    print(f'{name:12s}  acc={scores.mean():.5f} (+/-{scores.std():.5f})')

XGBoost       acc=0.98413 (+/-0.00020)
LightGBM-1    acc=0.98599 (+/-0.00045)
LightGBM-2    acc=0.98801 (+/-0.00039)


## train ensemble on full data
soft voting with the 3 models — no stacking CV needed, this is fast and works well

In [8]:
voting = VotingClassifier(
    estimators=[
        ('xgb', xgb1),
        ('lgbm1', lgbm1),
        ('lgbm2', lgbm2)
    ],
    voting='soft',
    n_jobs=-1
)

print('training ensemble...')
voting.fit(X, y)
print('done')

training ensemble...
done


## predict and save

In [9]:
preds = voting.predict(X_test)

print(f'total: {len(preds)}')
print(f'class 0 (normal): {(preds==0).sum()}')
print(f'class 1 (faulty): {(preds==1).sum()}')

total: 10944
class 0 (normal): 6668
class 1 (faulty): 4276


In [10]:
output = pd.DataFrame({
    'ID': test_ids,
    'CLASS': preds.astype(int)
})

output.to_csv('FINAL.csv', index=False)
print(f'saved FINAL.csv with {len(output)} rows')
output.head(10)

saved FINAL.csv with 10944 rows


,ID,CLASS
0,1,1
1,2,0
2,3,1
3,4,0
4,5,0
5,6,1
6,7,0
7,8,1
8,9,1
9,10,0


# ML Challenge - Fault Detection
binary classification on sensor data (47 features) to detect faulty vs normal devices

In [11]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## load data

In [12]:
train = pd.read_csv('TRAIN.csv')
test = pd.read_csv('TEST.csv')

print(train.shape, test.shape)
train.head()

(43776, 48) (10944, 48)


,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
0,0.185570,0.004568,0.005362,0.003335,0.005415,0.004895,0.012764,0.120138,0.140450,3.361753,...,0.041526,-0.230857,0.003310,0.042250,0.005975,0.002104,0.013878,0.001518,0.011347,0
1,0.369536,0.003983,0.003386,0.004902,0.007570,0.012136,0.118050,0.323925,0.132093,2.766117,...,-0.141285,-6.222857,0.834177,0.227968,0.018463,-0.020487,0.001246,0.007489,0.008724,1
2,0.602510,0.008442,0.012961,0.012870,0.046885,0.115401,0.065688,0.306677,0.498805,4.521201,...,0.011334,10.335251,-0.276614,-0.198900,-0.012756,0.014286,-0.001866,0.002687,0.013452,1
3,0.347957,0.064721,0.013611,0.011541,0.006492,0.008690,0.013192,0.164553,0.298665,3.170847,...,0.190479,2.864912,-1.921939,0.891690,1.108098,0.635431,0.081368,-0.000225,0.009166,0
4,0.233653,0.012217,0.010088,0.022095,0.026040,0.015062,0.016063,0.084648,0.213367,8.150943,...,0.203164,0.001812,-0.092731,0.005280,-0.213985,0.032195,0.002081,0.028930,-0.025912,1


In [13]:
# check class balance
train['Class'].value_counts()

Class
0    26465
1    17311
Name: count, dtype: int64

In [14]:
# any missing values?
print('train missing:', train.isnull().sum().sum())
print('test missing:', test.isnull().sum().sum())

train missing: 0
test missing: 0


In [15]:
train.describe()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
count,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,...,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000
mean,0.567525,0.014285,0.015551,0.026779,0.087637,0.155142,0.150656,0.255644,0.361146,3.762811,...,0.023477,-0.039795,0.005846,-0.000089,0.003215,0.001007,0.004599,0.003981,-0.006780,0.395445
std,0.738884,0.019607,0.022504,0.050725,0.188394,0.377655,0.339948,0.354816,0.440134,1.271343,...,0.707224,8.462749,1.631328,1.103486,0.354742,0.171935,0.201611,0.202130,0.562031,0.488952
min,0.100025,0.003001,0.000752,0.000682,0.000835,0.000854,0.000702,0.001399,0.001164,0.038349,...,-22.647806,-28.026880,-9.109769,-9.275399,-10.607320,-5.910117,-11.029342,-18.853842,-44.878769,0.000000
25%,0.198843,0.004778,0.005437,0.006290,0.012050,0.014112,0.015084,0.072098,0.128780,3.192588,...,-0.036589,-0.254039,-0.117789,-0.015759,-0.011839,-0.007214,-0.005226,-0.000826,-0.008882,0.000000
50%,0.288778,0.007988,0.008905,0.012619,0.023681,0.032396,0.029157,0.119095,0.214665,3.604579,...,-0.000210,-0.000757,-0.002891,-0.000111,-0.000045,-0.000170,0.000295,0.000092,-0.000083,0.000000
75%,0.528760,0.014382,0.016610,0.027599,0.063469,0.115950,0.116992,0.265227,0.381230,4.035824,...,0.043276,0.171122,0.107351,0.013399,0.009806,0.005892,0.006058,0.002405,0.007418,1.000000
max,12.779628,0.199925,0.506419,0.851009,5.017180,7.249545,6.556998,5.960009,5.085546,50.624777,...,89.378571,28.471415,10.907778,9.374269,11.399358,9.139964,18.623611,8.960172,40.818882,1.000000


## preprocessing

In [16]:
# drop duplicates from train
before = len(train)
train = train.drop_duplicates()
print(f'dropped {before - len(train)} duplicate rows, now {len(train)}')

dropped 738 duplicate rows, now 43038


In [17]:
test_ids = test['ID']
feature_cols = [c for c in train.columns if c != 'Class']
test_feature_cols = [c for c in test.columns if c not in ('ID', 'Class')]

print(f'features: {len(feature_cols)}')

features: 47


## feature engineering
adding some extra features - stats across rows, log transforms for the heavy tailed ones, and interaction terms between the most correlated features

In [18]:
def make_features(df, feat_cols):
    data = df[feat_cols].copy()
    
    # row-wise stats
    data['feat_mean'] = data.mean(axis=1)
    data['feat_std'] = data.std(axis=1)
    data['feat_min'] = data.min(axis=1)
    data['feat_max'] = data.max(axis=1)
    data['feat_median'] = data.median(axis=1)
    data['feat_range'] = data['feat_max'] - data['feat_min']
    data['feat_skew'] = data[feat_cols].skew(axis=1)
    data['feat_kurtosis'] = data[feat_cols].kurtosis(axis=1)
    
    # log transform the heavy-tailed features (F30-F38 are all positive)
    for col in ['F30','F31','F32','F33','F34','F35','F36','F37','F38']:
        data[f'{col}_log'] = np.log1p(data[col])
    
    # interactions between top correlated features with target
    data['F01_x_F09'] = data['F01'] * data['F09']
    data['F01_x_F29'] = data['F01'] * data['F29']
    data['F19_div_F21'] = data['F19'] / (data['F21'] + 1e-8)
    data['F05_div_F06'] = data['F05'] / (data['F06'] + 1e-8)
    data['F25_x_F27'] = data['F25'] * data['F27']
    data['F07_x_F08'] = data['F07'] * data['F08']
    
    # group sums
    data['sum_F01_F09'] = data[['F01','F02','F03','F04','F05','F06','F07','F08','F09']].sum(axis=1)
    data['sum_F10_F18'] = data[['F10','F11','F12','F13','F14','F15','F16','F17','F18']].sum(axis=1)
    data['sum_F19_F29'] = data[['F19','F20','F21','F22','F23','F24','F25','F26','F27','F28','F29']].sum(axis=1)
    data['sum_F30_F38'] = data[['F30','F31','F32','F33','F34','F35','F36','F37','F38']].sum(axis=1)
    data['sum_F39_F47'] = data[['F39','F40','F41','F42','F43','F44','F45','F46','F47']].sum(axis=1)
    
    return data

In [19]:
X_eng = make_features(train, feature_cols)
X_test_eng = make_features(test, test_feature_cols)

X = X_eng.values
y = train['Class'].values
X_test = X_test_eng.values

print(f'final feature count: {X.shape[1]}')
print(f'train samples: {X.shape[0]}, test samples: {X_test.shape[0]}')

final feature count: 75
train samples: 43038, test samples: 10944


## define models
going with xgboost + lightgbm + random forest, two variants each for the boosting models to get more diversity in the ensemble

In [20]:
scale_pos = y[y==0].shape[0] / y[y==1].shape[0]
print(f'scale_pos_weight = {scale_pos:.3f}')

scale_pos_weight = 1.486


In [21]:
xgb1 = XGBClassifier(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.6,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos,
    min_child_weight=5,
    gamma=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    tree_method='hist'
)

xgb2 = XGBClassifier(
    n_estimators=1200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.7,
    reg_alpha=0.05,
    reg_lambda=0.8,
    scale_pos_weight=scale_pos,
    min_child_weight=3,
    gamma=0.05,
    random_state=123,
    n_jobs=-1,
    eval_metric='logloss',
    tree_method='hist'
)

In [22]:
lgbm1 = LGBMClassifier(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.6,
    reg_alpha=0.1,
    reg_lambda=1.0,
    is_unbalance=True,
    min_child_samples=20,
    num_leaves=63,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm2 = LGBMClassifier(
    n_estimators=1200,
    max_depth=10,
    learning_rate=0.05,
    subsample=0.75,
    colsample_bytree=0.7,
    reg_alpha=0.05,
    reg_lambda=0.5,
    is_unbalance=True,
    min_child_samples=30,
    num_leaves=127,
    random_state=123,
    n_jobs=-1,
    verbose=-1
)

In [23]:
rf = RandomForestClassifier(
    n_estimators=800,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

## cross validation
checking how each model does individually first

In [24]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'XGBoost-1': xgb1,
    'XGBoost-2': xgb2,
    'LightGBM-1': lgbm1,
    'LightGBM-2': lgbm2,
    'RandomForest': rf
}

cv_results = {}
for name, model in models.items():
    acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    f1 = cross_val_score(model, X, y, cv=cv, scoring='f1', n_jobs=-1)
    cv_results[name] = acc.mean()
    print(f'{name:15s}  acc={acc.mean():.5f} (+/-{acc.std():.5f})  f1={f1.mean():.5f} (+/-{f1.std():.5f})')

XGBoost-1        acc=0.98834 (+/-0.00059)  f1=0.98542 (+/-0.00075)
XGBoost-2        acc=0.98734 (+/-0.00066)  f1=0.98417 (+/-0.00085)
LightGBM-1       acc=0.98908 (+/-0.00027)  f1=0.98635 (+/-0.00033)
LightGBM-2       acc=0.98975 (+/-0.00041)  f1=0.98719 (+/-0.00050)
RandomForest     acc=0.97848 (+/-0.00109)  f1=0.97271 (+/-0.00142)


## ensemble
try both stacking and soft voting to see which works better

In [25]:
# stacking with logistic regression as meta learner
stacking = StackingClassifier(
    estimators=[
        ('xgb1', xgb1),
        ('xgb2', xgb2),
        ('lgbm1', lgbm1),
        ('lgbm2', lgbm2),
        ('rf', rf)
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

stacking_scores = cross_val_score(stacking, X, y, cv=cv, scoring='accuracy', n_jobs=1)
print(f'Stacking  acc={stacking_scores.mean():.5f} (+/-{stacking_scores.std():.5f})')

Stacking  acc=0.98917 (+/-0.00038)


In [27]:
# soft voting
voting = VotingClassifier(
    estimators=[
        ('xgb1', xgb1),
        ('xgb2', xgb2),
        ('lgbm1', lgbm1),
        ('lgbm2', lgbm2),
        ('rf', rf)
    ],
    voting='soft',
    n_jobs=-1
)

voting_scores = cross_val_score(voting, X, y, cv=cv, scoring='accuracy', n_jobs=1)
print(f'Voting    acc={voting_scores.mean():.5f} (+/-{voting_scores.std():.5f})')

Voting    acc=0.98896 (+/-0.00023)


## pick the best and train on full data

In [28]:
all_scores = {
    **cv_results,
    'Stacking': stacking_scores.mean(),
    'Voting': voting_scores.mean()
}

best_name = max(all_scores, key=all_scores.get)
print(f'best: {best_name} with acc={all_scores[best_name]:.5f}')
print()
for k, v in sorted(all_scores.items(), key=lambda x: -x[1]):
    print(f'  {k:15s} {v:.5f}')

best: LightGBM-2 with acc=0.98975

  LightGBM-2      0.98975
  Stacking        0.98917
  LightGBM-1      0.98908
  Voting          0.98896
  XGBoost-1       0.98834
  XGBoost-2       0.98734
  RandomForest    0.97848


In [29]:
model_map = {
    'XGBoost-1': xgb1,
    'XGBoost-2': xgb2,
    'LightGBM-1': lgbm1,
    'LightGBM-2': lgbm2,
    'RandomForest': rf,
    'Stacking': stacking,
    'Voting': voting
}

final_model = model_map[best_name]
print(f'training {best_name} on all {X.shape[0]} samples...')
final_model.fit(X, y)
print('done')

training LightGBM-2 on all 43038 samples...
done


## predict and save

In [30]:
preds = final_model.predict(X_test)

print(f'predictions: {len(preds)}')
print(f'class 0: {(preds==0).sum()}')
print(f'class 1: {(preds==1).sum()}')

predictions: 10944
class 0: 6666
class 1: 4278


In [31]:
output = pd.DataFrame({
    'ID': test_ids,
    'CLASS': preds.astype(int)
})

output.to_csv('FINAL.csv', index=False)
print(f'saved FINAL.csv ({len(output)} rows)')
output.head(10)

saved FINAL.csv (10944 rows)


,ID,CLASS
0,1,1
1,2,0
2,3,1
3,4,0
4,5,0
5,6,1
6,7,0
7,8,1
8,9,1
9,10,0
